# Anima 引擎部署（Sprint 10：Worker API 轮询 + LLM 4 槽位容灾）

前置：Cloudflare Worker 已部署并配置 `ENGINE_KEY` secret。
本 notebook 在下方单元格填写 ①你的仓库地址（含改造后代码） ②Worker 地址 ③ENGINE_KEY ④LLM 4 槽位配置（第 1 个为主，出错按序切换；共用同一 model）。

In [ ]:
# ===== 必填配置 =====
REPO_URL = "https://github.com/<你的账号>/AnimaBot.git"   # 含 Sprint 10 改造后代码的仓库
WORKER_BASE_URL = "https://anima.example.com"              # 部署的 Worker 域名
ENGINE_KEY = ""                                            # 与 Worker `wrangler secret put ENGINE_KEY` 一致
ENGINE_ID = "engine-1"

# ===== LLM 配置（Sprint 10：4 个槽位，第 1 个为主，出错按序切换；共用同一模型） =====
# 每个槽位只需 api_key + base_url；model 全局一个（如 deepseek-v4-flash）
MODEL = "deepseek-v4-flash"
PROVIDERS = [
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
]

import os, json, subprocess, sys, time
os.environ["WORKER_BASE_URL"] = WORKER_BASE_URL
os.environ["ENGINE_KEY"] = ENGINE_KEY
os.environ["ENGINE_ID"] = ENGINE_ID

# 启动前校验：缺关键配置时立即提示，避免引擎刷 Illegal header / 连错域名
assert WORKER_BASE_URL not in ("", "http://127.0.0.1:8787", "https://anima.example.com"), \
    f"WORKER_BASE_URL 未填！应填你的 Worker 域名（当前={WORKER_BASE_URL}）"
assert len(ENGINE_KEY) >= 16, "ENGINE_KEY 未填！应与 Worker secret 一致（当前长度 %d）" % len(ENGINE_KEY)
print("配置已设置（WORKER_BASE_URL=", WORKER_BASE_URL, "，ENGINE_KEY 长度:", len(ENGINE_KEY), "，LLM 槽位数:", len(PROVIDERS), "）")

In [ ]:
!git clone https://github.com/chinokikiss/ComfyUI.git || true
!rm -rf /kaggle/working/AnimaBot && git clone {REPO_URL} /kaggle/working/AnimaBot

In [ ]:
!curl -L -o /kaggle/working/ComfyUI/models/diffusion_models/anima-base-v1.0.safetensors "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors?download=true"
!curl -L -o /kaggle/working/ComfyUI/models/text_encoders/qwen_3_06b_base.safetensors "https://huggingface.co/cnmds/man/resolve/main/qwen_3_06b_base.safetensors?download=true"
!curl -L -o /kaggle/working/ComfyUI/models/upscale_models/4x-AnimeSharp.safetensors "https://huggingface.co/cnmds/man/resolve/main/4x-AnimeSharp.safetensors?download=true"
!curl -L -o /kaggle/working/ComfyUI/models/vae/qwen_image_vae.safetensors "https://huggingface.co/cnmds/man/resolve/main/qwen_image_vae.safetensors?download=true"
!curl -L -o /kaggle/working/ComfyUI/models/loras/anima-turbo-lora-v0.2.safetensors "https://huggingface.co/cnmds/man/resolve/main/anima-turbo-lora-v0.2.safetensors?download=true"

In [ ]:
!pip install -r /kaggle/working/ComfyUI/requirements.txt -q
!pip install -r /kaggle/working/AnimaBot/requirements.txt -q
!pip install sageattention -q
!apt-get install -y oxipng > /dev/null 2>&1 || echo "oxipng 安装失败（可选，跳过压缩）"

In [ ]:
import json
cfg = {
  "providers": PROVIDERS,
  "model": MODEL,
  "logging": True,
}
with open("/kaggle/working/AnimaBot/config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
print("config.json 已写入（providers:", len(PROVIDERS), "）")

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Sprint 11 修复：ComfyUI main.py 不支持 --device 参数，改用 CUDA_VISIBLE_DEVICES 指定 GPU。
# 每个 ComfyUI 进程一个独立子环境（CUDA_VISIBLE_DEVICES=0 / =1），端口 8188 / 8189。
# 引擎 core.py 不在子进程里指定 GPU（保持默认可见），ComfyUI 由引擎自动选空闲实例。
log_dir = Path('/kaggle/working/engine_logs')
log_dir.mkdir(exist_ok=True)

tasks = [
    {"script": "/kaggle/working/ComfyUI/main.py", "cwd": "/kaggle/working/ComfyUI",
     "env": {"CUDA_VISIBLE_DEVICES": "0"},
     "args": ["--disable-cuda-malloc", "--use-sage-attention", "--disable-dynamic-vram", "--gpu-only", "--port", "8188"],
     "out": str(log_dir / "comfyui-8188.log"), "err": str(log_dir / "comfyui-8188.err")},
    {"script": "/kaggle/working/ComfyUI/main.py", "cwd": "/kaggle/working/ComfyUI",
     "env": {"CUDA_VISIBLE_DEVICES": "1"},
     "args": ["--disable-cuda-malloc", "--use-sage-attention", "--disable-dynamic-vram", "--gpu-only", "--port", "8189"],
     "out": str(log_dir / "comfyui-8189.log"), "err": str(log_dir / "comfyui-8189.err")},
    {"script": "/kaggle/working/AnimaBot/core.py", "cwd": "/kaggle/working/AnimaBot", "args": [],
     "out": str(log_dir / "engine.log"), "err": str(log_dir / "engine.err")},
]

base_env = dict(os.environ)
base_env.update({"WORKER_BASE_URL": WORKER_BASE_URL, "ENGINE_KEY": ENGINE_KEY, "ENGINE_ID": ENGINE_ID})

for t in tasks:
    out = open(t.get("out", os.devnull), "w")
    err = open(t.get("err", os.devnull), "w")
    proc_env = dict(base_env)
    proc_env.update(t.get("env", {}))  # 每进程追加 CUDA_VISIBLE_DEVICES 等子环境
    # 引擎 core.py 加 -u 标志：禁用 Python 输出缓冲，日志实时写入文件（之前 engine.log 为 0KB 是缓冲导致的）
    p = subprocess.Popen([sys.executable, "-u", t["script"]] + t["args"], cwd=t["cwd"], env=proc_env,
                         stdout=out, stderr=err)
    print("已启动:", t["script"], "PID", p.pid, "| env", t.get("env", {}))

print("\n全部进程已在后台启动（无 NapCat）")
print("引擎日志: /kaggle/working/engine_logs/engine.log （tail -u 查看实时）")
print("错误明细: /kaggle/working/engine_logs/errors.log （每次任务失败追加完整步骤日志）")
print("ComfyUI: comfyui-8188.log / comfyui-8189.log（CUDA 0/1，两个实例，引擎自动就绪重试）")